In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar datos de la Capa Silver
df_silver = spark.table("telemetria_patio_silver")

# ======================================================
# 2. CÁLCULO DE TIEMPOS ENTRE EVENTOS (Lógica de Negocio)
# ======================================================
# Usamos Window Functions para calcular la diferencia de tiempo con el siguiente evento
window_nia = Window.partitionBy("logs_nia").orderBy("evento_ts")

df_tiempos = df_silver.withColumn(
    "evento_ts_siguiente", 
    F.lead("evento_ts").over(window_nia)
).withColumn(
    "tiempo_min", 
    (F.col("evento_ts_siguiente") - F.col("evento_ts")) / 1000 / 60
)

# Filtrar eventos finales (donde no hay siguiente evento) y duraciones negativas
df_tiempos = df_tiempos.filter(F.col("tiempo_min") >= 0)

# ======================================================
# 3. CREACIÓN DE TABLA DE HECHOS: Fact_Movimientos
# ======================================================
# Esta tabla contiene cada "salto" entre zonas y su duración
fact_movimientos = df_tiempos.select(
    F.col("logs_nia").alias("id_nia"),
    F.col("evento_fecha_lima").alias("fecha_inicio"),
    F.col("logs_ubicacion").alias("zona"),
    F.round(F.col("tiempo_min"), 2).alias("duracion_minutos"),
    F.col("shared_placaTracto").alias("id_vehiculo"),
    F.col("shared_conductor").alias("id_conductor")
)

# Guardar Fact Table
fact_movimientos.write.format("delta").mode("overwrite").saveAsTable("gold_patio_fact_movimientos")

# ======================================================
# 4. CREACIÓN DE DIMENSIONES (MODELO ESTRELLA)
# ======================================================

# Dim_Vehiculo: Atributos únicos de los activos
dim_vehiculo = df_silver.select(
    F.col("shared_placaTracto").alias("placa"),
    F.col("shared_tipo").alias("tipo_vehiculo"),
    F.col("shared_placaPlataforma").alias("placa_carreta")
).distinct()

# Dim_Conductor: Atributos de los choferes y empresas
dim_conductor = df_silver.select(
    F.col("shared_conductor").alias("nombre_conductor"),
    F.col("shared_empresa").alias("empresa_transporte")
).distinct()

# Guardar Dimensiones
dim_vehiculo.write.format("delta").mode("overwrite").saveAsTable("gold_patio_dim_vehiculo")
dim_conductor.write.format("delta").mode("overwrite").saveAsTable("gold_patio_dim_conductor")

print("🏆 CAPA GOLD FINALIZADA: Modelo Estrella disponible para Power BI.")

In [0]:
%sql
SELECT * FROM gold_patio_fact_movimientos 
ORDER BY fecha_inicio DESC 
LIMIT 100